# Laboratorio E: MNIST con TensorFlow + Keras (sin CNN)

**Redes Neuronales II - Especializacion en Inteligencia Artificial**
**Autor:** Harold Steven Rueda Lara
**Objetivo:** Clasificar digitos escritos a mano (0-9) usando una red densa (Flatten + Dense), sin ninguna capa convolucional, aplicando la misma logica multiclase del Laboratorio D a un problema real de imagenes.

> **Importante:** este notebook debe ejecutarse en Google Colab (Entorno de ejecucion > Ejecutar todas) para descargar el dataset MNIST completo (70.000 imagenes) y generar las evidencias de ejecucion (capturas) que van en el documento tecnico.


## Paso 26. Cargar MNIST

In [ ]:
from tensorflow import keras
import numpy as np

(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = keras.datasets.mnist.load_data()
print("Entrenamiento:", x_train_mnist.shape, y_train_mnist.shape)
print("Prueba:", x_test_mnist.shape, y_test_mnist.shape)

**Resultado esperado:** 60.000 imagenes de entrenamiento y 10.000 de prueba, cada una de 28x28 pixeles.

## Paso 27. Ver una imagen y normalizar
Normalizar convierte los pixeles de la escala 0-255 a 0-1; la red no "ve" el digito, procesa valores numericos de pixeles.

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(x_train_mnist[0], cmap="gray")
plt.title(f"Numero correcto: {y_train_mnist[0]}")
plt.axis("off")
plt.show()

x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0
print("Minimo:", x_train_mnist.min())
print("Maximo:", x_train_mnist.max())

## Paso 28. Crear arquitectura 784 -> 128 -> 10
`Flatten` convierte la imagen de 28x28 en un vector de 784 valores. La capa final tiene 10 neuronas (una por digito). **No se usa ninguna capa convolucional.**

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)

modelo_mnist = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])
modelo_mnist.summary()

## Paso 29. Compilar y entrenar

In [ ]:
modelo_mnist.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
historial_mnist = modelo_mnist.fit(
    x_train_mnist, y_train_mnist, epochs=5, validation_split=0.1
)

## Paso 30. Graficar loss y accuracy

In [ ]:
plt.plot(historial_mnist.history["loss"], label="Entrenamiento")
plt.plot(historial_mnist.history["val_loss"], label="Validacion")
plt.title("Error durante el entrenamiento")
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.plot(historial_mnist.history["accuracy"], label="Entrenamiento")
plt.plot(historial_mnist.history["val_accuracy"], label="Validacion")
plt.title("Precision durante el entrenamiento")
plt.xlabel("Epoca")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Paso 31. Evaluar y predecir

In [ ]:
test_loss_mnist, test_accuracy_mnist = modelo_mnist.evaluate(
    x_test_mnist, y_test_mnist, verbose=0
)
print("Error de prueba:", test_loss_mnist)
print("Accuracy de prueba:", test_accuracy_mnist)

predicciones_mnist = modelo_mnist.predict(x_test_mnist[:10], verbose=0)
for i in range(10):
    prediccion = np.argmax(predicciones_mnist[i])
    print("Imagen", i + 1, "| Real:", y_test_mnist[i], "| Prediccion:", prediccion)

## Paso 32. Buscar un error del modelo
Un accuracy alto no significa perfeccion; analizar al menos un error ayuda a entender las limitaciones del modelo.

In [ ]:
predicciones_todas = modelo_mnist.predict(x_test_mnist, verbose=0)
clases_predichas = np.argmax(predicciones_todas, axis=1)
errores = np.where(clases_predichas != y_test_mnist)[0]
print("Cantidad de errores:", len(errores))

indice_error = errores[0]
plt.imshow(x_test_mnist[indice_error], cmap="gray")
plt.title(
    f"Real: {y_test_mnist[indice_error]} | "
    f"Prediccion: {clases_predichas[indice_error]}"
)
plt.axis("off")
plt.show()

## Conclusion del laboratorio
Con una arquitectura puramente densa (sin CNN), la red logra clasificar digitos MNIST con alta precision, confirmando que Flatten + Dense + Softmax es suficiente para este problema, aunque las redes convolucionales suelen superarla en datasets de imagenes mas complejos.